# Calculating an XMY

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
from climakitae.util.utils import (
    convert_to_local_time,
    get_closest_gridcell,
)
from climakitae.core.data_export import write_tmy_file
from climakitae.core.data_interface import get_data
#from climakitaegui.explore.typical_meteorological_year import plot_one_var_cdf

import pandas as pd
import xarray as xr
import numpy as np
import pkg_resources
from tqdm.auto import tqdm  # Progress bar

import warnings
warnings.filterwarnings("ignore")

### Step 1: Grab and process all required input data

The [TMY3 method](https://www.nrel.gov/docs/fy08osti/43156.pdf) selects a "typical" month based on ten daily variables: max, min, and mean air and dew point temperatures, max and mean wind speed, global irradiance and direct irradiance.  

#### Step 1a: Select location of interest
TMYs are calculated for a specific location of interest, like a building or power plant. Here, we will use a known weather station location, via their latitude and longitude to extract the data that we need to calculate the TMY. In the example below, we will look specifically at Los Angeles International Airport, but will note in the code below how you can provide your own location coordinates too. 

In [2]:
# read in station file of CA HadISD stations
stn_file = pkg_resources.resource_filename("climakitae", "data/hadisd_stations.csv")
stn_file = pd.read_csv(stn_file, index_col=[0])
stn_file.head(5)

,state,station,city,ID,LAT_Y,LON_X,station id,elevation
0,CA,Bakersfield Meadows Field (KBFL),Bakersfield,KBFL,35.43424,-119.05524,72384023155,149.3
1,CA,Blythe Asos (KBLH),Blythe,KBLH,33.61876,-114.71451,74718823158,120.4
2,CA,Burbank-Glendale-Pasadena Airport (KBUR),Burbank,KBUR,34.19966,-118.36543,72288023152,222.7
3,CA,Needles Airport (KEED),Needles,KEED,34.76783,-114.61842,72380523179,270.6
4,CA,Fresno Yosemite International Airport (KFAT),Fresno,KFAT,36.77999,-119.72016,72389093193,101.9


In [3]:
# grab airport
stn_name = "Bakersfield Meadows Field (KBFL)"
stn_code = stn_file.loc[stn_file["station"] == stn_name]["station id"].item()
one_station = stn_file.loc[stn_file["station"] == stn_name]
stn_lat = one_station.LAT_Y.item()
stn_lon = one_station.LON_X.item()
stn_state = one_station.state.item()
stn_lat, stn_lon

(35.43424, -119.05524)

In [4]:
print(stn_code)

72384023155


Alternatively, you may want to provide your own location instead of one of the HadISD stations above. If so, uncomment the cell below by removing the `#` symbol and modifying the lines of code. Note, with custom locations, if an elevation is not provided, a default value of 0.0 m will be used. 

In [5]:
## provide your own location, via latitude and longitude coordinates
# stn_lat = YOUR_LAT_HERE
# stn_lon = YOUR_LON_HERE
# stn_state = 'YOUR_STATE_HERE'
# stn_name = 'YOUR_STATION_NAME_HERE'
# stn_code = 'custom'
# stn_elev = YOUR_ELEV_HERE # meters

#### Step 1b: Select time frame of interest
The second required input for generating a TMY dataset is the **time frame of interest**. The recommended minimum number of input years for a TMY dataset is 15-20 years worth of daily data; we will use 30 years to represent a standard climatological period. For data post-2014, we will utilize SSP 3-7.0, although scenario selection in the near-future is relatively independent. If calculating a TMY for the far-future, **carefully consider which scenario SSP to include**, as there will be **significant** differences present. 

We will also process the data for our designated station location (latitude, and longitude) at 3 km over the <span style="color:#FF0000">1990-2020 period</span> as an example. **Note**: An additional year (2021) is also loaded to pad the end of the dataset after converting to local time in the next few steps -- this is done for you when subsetting for the data. 

In [6]:
# selected reference period
start_year = 1990
end_year = 2020

#### Step 1c: Retrieve variables in catalog
It is important to note that not all models in the Cal-Adapt: Analytics Engine have the solar variables critical for TMY file generation - in fact, only 4 do! We will carefully subset our variables to ensure that the same 4 models are selected for consistency. 

Lastly, because the dynamically downscaled WRF data in the Cal-Adapt: Analytics Engine is in UTC time, we'll convert to the timezone of the station we've selected. This is particularly important for the timing of solar radiation max and min, which is a critical piece of a TMY. The handy `convert_to_local_time` function corrects for this, and ensures that all data are within the same daily timestamp.

In [29]:
# selected models
data_models = [
    "WRF_EC-Earth3_r1i1p1f1",
    "WRF_MPI-ESM1-2-HR_r3i1p1f1",
    # "WRF_TaiESM1_r1i1p1f1",
    # "WRF_MIROC6_r1i1p1f1",
]

Now that we have set up the model list, let's start retrieving data! We will need to calculate summary statistics and reduce to daily timescales for the following variables:

In [8]:
# air temperature
temp_data = get_data(
    variable="Air Temperature at 2m",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    units="degC",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="Yes",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)

temp_data = convert_to_local_time(
    temp_data, stn_lon, stn_lat
)  # convert to local timezone, provide lon/lat because area average data lacks coordinates
temp_data = temp_data.sel({"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")})
temp_data = temp_data.sel(simulation=data_models)

# max air temp
max_airtemp_data = temp_data.resample(time="1D").max()  # daily max air temp
max_airtemp_data.name = "Daily max air temperature"  # rename for clarity

# min air temp
min_airtemp_data = temp_data.resample(time="1D").min()  # daily min air temp
min_airtemp_data.name = "Daily min air temperature"  # rename for clarity

# mean air temp
mean_airtemp_data = temp_data.resample(time="1D").mean()  # daily mean air temp
mean_airtemp_data.name = "Daily mean air temperature"  # rename for clarity
mean_airtemp_data = mean_airtemp_data.compute()

Data converted to America/Los_Angeles timezone.


Retrieve and calculate max and mean wind speed:

In [9]:
# wind speed
ws_data = get_data(
    variable="Wind speed at 10m",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    units="m s-1",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="Yes",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)

ws_data = convert_to_local_time(
    ws_data, stn_lon, stn_lat
)  # convert to local timezone, provide lon/lat because area average data lacks coordinates
ws_data = ws_data.sel(
    {"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")}
)  # your desired time slice in local time
ws_data = ws_data.sel(simulation=data_models)

# max wind speed
max_windspd_data = ws_data.resample(time="1D").max()  # daily max wind speed
max_windspd_data.name = "Daily max wind speed"  # rename for clarity

# mean wind speed
mean_windspd_data = ws_data.resample(time="1D").mean()  # daily mean wind speed
mean_windspd_data.name = "Daily mean wind speed"  # rename for clarity

Data converted to America/Los_Angeles timezone.


Retrieve and calculate max, min, and mean dew point temperature:

In [10]:
# dew point temperature
dewpt_data = get_data(
    variable="Dew point temperature",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    units="degC",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="Yes",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)

dewpt_data = convert_to_local_time(
    dewpt_data, stn_lon, stn_lat
)  # convert to local timezone, provide lon/lat because area average data lacks coordinates
dewpt_data = dewpt_data.sel({"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")})
dewpt_data = dewpt_data.sel(simulation=data_models)

# max dew point
max_dewpt_data = dewpt_data.resample(time="1D").max()  # daily max dewpoint temp
max_dewpt_data.name = "Daily max dewpoint temperature"  # rename for clarity

# min dew point
min_dewpt_data = dewpt_data.resample(time="1D").min()  # daily min dewpoint temp
min_dewpt_data.name = "Daily min dewpoint temperature"  # rename for clarity

# mean dew point
mean_dewpt_data = dewpt_data.resample(time="1D").mean()  # daily mean dewpoint temp
mean_dewpt_data.name = "Daily mean dewpoint temperature"  # rename for clarity

Data converted to America/Los_Angeles timezone.


Next, retrieve global horizontal irradiance. GHI is within the Analytics Engine catalog at daily resolutions, but for the TMY methodology, we need to calculate the total accumulated GHI received over the course of the day, so we will retrieve hourly data instead and calculate the necessary information below. The same process is repeated for the direct normal irradiance. 

In [11]:
# global irradiance
global_irradiance_data = get_data(
    variable="Instantaneous downwelling shortwave flux at bottom",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="Yes",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)

global_irradiance_data = convert_to_local_time(
    global_irradiance_data, stn_lon, stn_lat
)  # convert to local timezone, provide lon/lat because area average data lacks coordinates
global_irradiance_data = global_irradiance_data.sel(
    {"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")}
)
global_irradiance_data = global_irradiance_data.sel(simulation=data_models)

global_irradiance_data.name = "Global horizontal irradiance"  # rename for clarity
# total global horizontal irradiance (accumulation of hourly data over a 24-hour period)
total_ghi_data = global_irradiance_data.resample(time="1D").sum()

Data converted to America/Los_Angeles timezone.


In [12]:
# direct normal irradiance
direct_irradiance_data = get_data(
    variable="Shortwave surface downward direct normal irradiance",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="Yes",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)

direct_irradiance_data = convert_to_local_time(
    direct_irradiance_data, stn_lon, stn_lat
)  # convert to local timezone, provide lon/lat because area average data lacks coordinates
direct_irradiance_data = direct_irradiance_data.sel(
    {"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")}
)
direct_irradiance_data = direct_irradiance_data.sel(simulation=data_models)

direct_irradiance_data.name = "Direct normal irradiance"  # rename for clarity
# total direct normal irradiance (accumulation of hourly data over a 24-hour period)
total_dni_data = direct_irradiance_data.resample(time="1D").sum()

Data converted to America/Los_Angeles timezone.


#### Step 1d: Load all variables
Now that we have all of our data retrieved and calculated, it is time to actually load the data into memory. Previously, xarray has lazily loaded the data, but we will actually grab it now. This will take approximately **7 minutes**. 

In [13]:
all_vars = xr.merge(
    [
        max_airtemp_data.squeeze(),
        min_airtemp_data.squeeze(),
        mean_airtemp_data.squeeze(),
        max_dewpt_data.squeeze(),
        min_dewpt_data.squeeze(),
        mean_dewpt_data.squeeze(),
        max_windspd_data.squeeze(),
        mean_windspd_data.squeeze(),
        total_ghi_data.squeeze(),
        total_dni_data.squeeze(),
    ]
)

In [14]:
# load all indices in
all_vars = all_vars.compute()

In [15]:
#! export so I don't have to compute all vars everytine - takes quite a while to compute usually.
# though sometimes as little as 12 minutes. Why? At present, I do not know.
# Save
all_vars.to_netcdf("all_vars.nc")

### Step 2: Calculate percentile-based profile
Essentially, we are applying the standard profile methdology to a TMY.

In [7]:
# load saved all_vars eagerly
all_vars_import = xr.load_dataset("all_vars.nc")

##### Step 2a: Calculate percentiles

In [8]:
# scraps

# # Find the index of the value closest to the quantile
# closest_idx = abs(stacked - target_quantile).argmin(dim="all_dims")
# # Return the actual data value at that index
# closest_value = xr.DataArray(stacked.isel(all_dims=closest_idx).values)


In [9]:
def _closest_to_quantile(dat: xr.DataArray) -> xr.DataArray:
    """Find the actual data value closest to the specified quantile."""
    # Stack all dimensions except time_delta into a single dimension
    stacked = dat.stack(all_dims=list(dat.dims))
    # Compute the target quantile value
    target_quantile = stacked.quantile(0.5, dim="all_dims")

    return target_quantile

def get_percentile_by_sim(da):
    # Group the DataArray by simulation
    return da.groupby("simulation").apply(_closest_to_quantile)

def get_percentile_by_mon_and_sim(da):
    # Group the DataArray by month in the year
    return da.groupby("time.month").apply(get_percentile_by_sim)


#
def get_percentile_mon_yr(da):
        return da.groupby("time.year").apply(get_percentile_by_mon_and_sim)

def get_percentile_all(ds):
    """ get percentile by month and year"""
    return ds.apply(get_percentile_mon_yr)


#
def get_percentile_base(ds):
    """get percentile by month"""
    return ds.apply(get_percentile_by_mon_and_sim)

##### Optimize

In [64]:
# calculate per month percentile
# this gives us the extreme behavior across a single year timespan, derived from the 30 years of data

perc_fast = all_vars_import.groupby("time.month").quantile(0.5, dim="time")

In [71]:
perc_another = all_vars_import.apply(
    lambda da: da.groupby("time.month").apply(
        lambda month_da: month_da.groupby("simulation").apply(
            lambda sim_da: sim_da.stack(all_dims=list(sim_da.dims)).quantile(
                0.5, dim="all_dims"
            )
        )
    )
)

In [88]:
perc = get_percentile_base(all_vars_import)

['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulatio

In [69]:
perc_monthly_fast = all_vars_import.groupby("time.month").groupby("time.year").quantile(0.5, dim="time")

AttributeError: 'DatasetGroupBy' object has no attribute 'groupby'

In [68]:
perc_monthly_fast

<DatasetGroupBy, grouped over 1 grouper(s), 12 groups in total:
    'month': 12/12 groups present with labels 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12>

Perc ALL

In [ ]:
perc_monthly_fast = (
    all_vars_import.assign_coords(year=all_vars_import.time.dt.year, month=ds.time.dt.month)
    .stack(all_dims=["year", "month", "simulation"])
    .quantile(0.5, dim="all_dims")
)

In [91]:
perc_monthly_fast_2 = all_vars_import.stack(all_dims=["time", "simulation"]).quantile(0.5, dim="all_dims")

In [94]:
perc_monthly_fast_2

<xarray.Dataset> Size: 464B
Dimensions:                          (x: 5, y: 4)
Coordinates:
  * x                                (x) float64 40B -4.095e+06 ... -4.083e+06
  * y                                (y) float64 32B 1.01e+06 ... 1.019e+06
    lat                              (y, x) float32 80B 35.37 35.38 ... 35.5
    lon                              (y, x) float32 80B -119.1 -119.0 ... -119.0
    quantile                         float64 8B 0.5
Data variables:
    Daily max air temperature        float64 8B 26.09
    Daily min air temperature        float64 8B 13.25
    Daily mean air temperature       float64 8B 19.37
    Daily max dewpoint temperature   float64 8B 6.226
    Daily min dewpoint temperature   float64 8B -0.1008
    Daily mean dewpoint temperature  float64 8B 3.28
    Daily max wind speed             (y, x) float32 80B nan nan ... 4.486 nan
    Daily mean wind speed            (y, x) float32 80B nan nan ... 2.488 nan
    Global horizontal irradiance     float64 8B 6.04e+03
    Direct normal irradiance         float64 8B 9.096e+03
Attributes:
    variable_id:           t2
    extended_description:  Temperature of the air 2m above Earth's surface. T...
    units:                 degC
    data_type:             Gridded
    resolution:            3 km
    frequency:             hourly
    location_subset:       coordinate selection
    approach:              Time
    downscaling_method:    Dynamical
    institution:           UCLA
    grid_mapping:          Lambert_Conformal
    timezone:              America/Los_Angeles

In [92]:
perc_monthly = get_percentile_all(all_vars_import)

['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulation', 'time']
['simulatio

##### Run it

In [10]:
perc_base = get_percentile_base(all_vars_import)
perc_monthly = get_percentile_all(all_vars_import)

##### Step 2b: Drop volcano years (Pinatubo 1991-1994)

In [11]:
# Remove the years for the Pinatubo eruption
perc_monthly_dropped = perc_monthly.where(
    (~perc_monthly.year.isin([1991, 1992, 1993, 1994])), np.nan, drop=True
)

In [12]:
perc_monthly_dropped

<xarray.Dataset> Size: 52kB
Dimensions:                          (simulation: 2, year: 27, month: 12)
Coordinates:
    quantile                         float64 8B 0.5
  * simulation                       (simulation) <U26 208B 'WRF_EC-Earth3_r1...
  * month                            (month) int64 96B 1 2 3 4 5 ... 9 10 11 12
  * year                             (year) int64 216B 1990 1995 ... 2019 2020
Data variables:
    Daily max air temperature        (simulation, year, month) float64 5kB 15...
    Daily min air temperature        (simulation, year, month) float64 5kB 5....
    Daily mean air temperature       (simulation, year, month) float64 5kB 10...
    Daily max dewpoint temperature   (simulation, year, month) float64 5kB 4....
    Daily min dewpoint temperature   (simulation, year, month) float64 5kB -0...
    Daily mean dewpoint temperature  (simulation, year, month) float64 5kB 2....
    Daily max wind speed             (simulation, year, month) float64 5kB 3....
    Daily mean wind speed            (simulation, year, month) float64 5kB 1....
    Global horizontal irradiance     (simulation, year, month) float64 5kB 3....
    Direct normal irradiance         (simulation, year, month) float64 5kB 7....
Attributes:
    variable_id:           t2
    extended_description:  Temperature of the air 2m above Earth's surface. T...
    units:                 degC
    data_type:             Gridded
    resolution:            3 km
    frequency:             hourly
    location_subset:       coordinate selection
    approach:              Time
    downscaling_method:    Dynamical
    institution:           UCLA
    grid_mapping:          Lambert_Conformal
    timezone:              America/Los_Angeles

### Step 3: Variable weighting according to NREL methods

#### Step 3a: Calculate the Finkelstein-Schafer statistic 
The [Finkelstein-Schafer statistic](https://academic.oup.com/biomet/article-abstract/58/3/641/233677) determines the absolute difference between the long-term climatology and candidate CDF profiles, and considers the number of days within each month. We will use a handy function `fs_statistic` to calculate this below. 

In [13]:
def fs_statistic(cdf_climatology, cdf_month):
    """
    Calculates the Finkelstein-Schafer statistic:
    Absolute difference between quantile and candidate quantiles, divided by number of days in month
    """
    days_per_mon = xr.DataArray(
        data=[31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31],
        coords={"month": np.arange(1, 13)},
    )
    #fs_stat = abs(cdf_month - cdf_climatology).sel(data="probability") / days_per_mon
    fs_stat = abs(cdf_month - cdf_climatology) / days_per_mon
    return fs_stat

In [14]:
all_vars_fs = fs_statistic(perc_base, perc_monthly)

In [15]:
all_vars_fs

<xarray.Dataset> Size: 60kB
Dimensions:                          (simulation: 2, month: 12, year: 31)
Coordinates:
    quantile                         float64 8B 0.5
  * simulation                       (simulation) <U26 208B 'WRF_EC-Earth3_r1...
  * month                            (month) int64 96B 1 2 3 4 5 ... 9 10 11 12
  * year                             (year) int64 248B 1990 1991 ... 2019 2020
Data variables:
    Daily max air temperature        (simulation, year, month) float64 6kB 0....
    Daily min air temperature        (simulation, year, month) float64 6kB 0....
    Daily mean air temperature       (simulation, year, month) float64 6kB 0....
    Daily max dewpoint temperature   (simulation, year, month) float64 6kB 0....
    Daily min dewpoint temperature   (simulation, year, month) float64 6kB 0....
    Daily mean dewpoint temperature  (simulation, year, month) float64 6kB 0....
    Daily max wind speed             (simulation, year, month) float64 6kB 0....
    Daily mean wind speed            (simulation, year, month) float64 6kB 0....
    Global horizontal irradiance     (simulation, year, month) float64 6kB 6....
    Direct normal irradiance         (simulation, year, month) float64 6kB 55...
Attributes:
    variable_id:           t2
    extended_description:  Temperature of the air 2m above Earth's surface. T...
    units:                 degC
    data_type:             Gridded
    resolution:            3 km
    frequency:             hourly
    location_subset:       coordinate selection
    approach:              Time
    downscaling_method:    Dynamical
    institution:           UCLA
    grid_mapping:          Lambert_Conformal
    timezone:              America/Los_Angeles

#### Step 3b: Weight the F-S statistic

Next, we weight the F-S statistic results based on the input variables. The [TMY3](https://www.nrel.gov/docs/fy08osti/43156.pdf) method places a higher weight on the solar variables (global irradiance and direct irradiance), which we follow here. 

In [16]:
def compute_weighted_fs(da_fs):
    """Weights the F-S statistics based on TMY3 methodology"""
    weights_per_var = {
        "Daily max air temperature": 1 / 20,
        "Daily min air temperature": 1 / 20,
        "Daily mean air temperature": 2 / 20,
        "Daily max dewpoint temperature": 1 / 20,
        "Daily min dewpoint temperature": 1 / 20,
        "Daily mean dewpoint temperature": 2 / 20,
        "Daily max wind speed": 1 / 20,
        "Daily mean wind speed": 1 / 20,
        "Global horizontal irradiance": 5 / 20,
        "Direct normal irradiance": 5 / 20,
    }

    for var, weight in weights_per_var.items():
        # Multiply each variable by it's appropriate weight
        da_fs[var] = da_fs[var] * weight
    return da_fs

In [17]:
weighted_fs = compute_weighted_fs(all_vars_fs)

In [18]:
weighted_fs

<xarray.Dataset> Size: 60kB
Dimensions:                          (simulation: 2, month: 12, year: 31)
Coordinates:
    quantile                         float64 8B 0.5
  * simulation                       (simulation) <U26 208B 'WRF_EC-Earth3_r1...
  * month                            (month) int64 96B 1 2 3 4 5 ... 9 10 11 12
  * year                             (year) int64 248B 1990 1991 ... 2019 2020
Data variables:
    Daily max air temperature        (simulation, year, month) float64 6kB 0....
    Daily min air temperature        (simulation, year, month) float64 6kB 0....
    Daily mean air temperature       (simulation, year, month) float64 6kB 0....
    Daily max dewpoint temperature   (simulation, year, month) float64 6kB 0....
    Daily min dewpoint temperature   (simulation, year, month) float64 6kB 0....
    Daily mean dewpoint temperature  (simulation, year, month) float64 6kB 0....
    Daily max wind speed             (simulation, year, month) float64 6kB 0....
    Daily mean wind speed            (simulation, year, month) float64 6kB 0....
    Global horizontal irradiance     (simulation, year, month) float64 6kB 1....
    Direct normal irradiance         (simulation, year, month) float64 6kB 13...
Attributes:
    variable_id:           t2
    extended_description:  Temperature of the air 2m above Earth's surface. T...
    units:                 degC
    data_type:             Gridded
    resolution:            3 km
    frequency:             hourly
    location_subset:       coordinate selection
    approach:              Time
    downscaling_method:    Dynamical
    institution:           UCLA
    grid_mapping:          Lambert_Conformal
    timezone:              America/Los_Angeles

##### Step 3c: Select candidate months for consideration

In [19]:
weighted_fs

<xarray.Dataset> Size: 60kB
Dimensions:                          (simulation: 2, month: 12, year: 31)
Coordinates:
    quantile                         float64 8B 0.5
  * simulation                       (simulation) <U26 208B 'WRF_EC-Earth3_r1...
  * month                            (month) int64 96B 1 2 3 4 5 ... 9 10 11 12
  * year                             (year) int64 248B 1990 1991 ... 2019 2020
Data variables:
    Daily max air temperature        (simulation, year, month) float64 6kB 0....
    Daily min air temperature        (simulation, year, month) float64 6kB 0....
    Daily mean air temperature       (simulation, year, month) float64 6kB 0....
    Daily max dewpoint temperature   (simulation, year, month) float64 6kB 0....
    Daily min dewpoint temperature   (simulation, year, month) float64 6kB 0....
    Daily mean dewpoint temperature  (simulation, year, month) float64 6kB 0....
    Daily max wind speed             (simulation, year, month) float64 6kB 0....
    Daily mean wind speed            (simulation, year, month) float64 6kB 0....
    Global horizontal irradiance     (simulation, year, month) float64 6kB 1....
    Direct normal irradiance         (simulation, year, month) float64 6kB 13...
Attributes:
    variable_id:           t2
    extended_description:  Temperature of the air 2m above Earth's surface. T...
    units:                 degC
    data_type:             Gridded
    resolution:            3 km
    frequency:             hourly
    location_subset:       coordinate selection
    approach:              Time
    downscaling_method:    Dynamical
    institution:           UCLA
    grid_mapping:          Lambert_Conformal
    timezone:              America/Los_Angeles

In [22]:
# Sum
weighted_fs_sum = weighted_fs.to_array().sum(dim=["variable"]).drop(["quantile"])

In [23]:
weighted_fs_sum

<xarray.DataArray (simulation: 2, year: 31, month: 12)> Size: 6kB
array([[[15.36789408, 14.04620053,  8.99526736, 13.10756905,
          1.23171175,  0.91893894,  1.07509372,  9.07839516,
          0.36350244,  0.95801438,  7.80660883,  2.79112221],
        [22.7793316 , 22.05611877, 28.43632715, 34.43826107,
          2.89726535,  0.72721037,  4.73851567,  1.48893447,
          0.83228296,  5.21219046, 34.14475159, 10.19195826],
        [12.33266146,  6.97291323,  7.00290922, 10.22289548,
          1.95168965,  0.65947625,  2.55523744,  0.52858353,
          2.1521432 ,  2.8248426 ,  4.46596807,  3.6781029 ],
        [20.01602906, 18.22379496,  4.79926404,  7.89341606,
          2.67966456,  9.01018781,  1.58051696,  2.45303135,
          2.83440019,  0.33877976,  6.88847823, 11.51717323],
        [12.56666181, 15.45113099,  9.62514375, 12.64991123,
          1.09542145,  0.39063545,  0.11103129,  0.11530619,
          3.71755442,  8.78165231,  2.50261948, 15.58330323],
        [ 9.47540692, 26.66715908, 24.49129426, 11.09928995,
          3.42583241,  0.12614735,  1.7781783 ,  2.27451445,
          3.14949208,  3.37963708, 24.27839261, 35.53486726],
        [ 7.25888264, 27.28818454,  5.68240129,  7.79313271,
          2.1388389 ,  2.54800082,  0.87806062,  7.3820356 ,
...
          2.09266998,  1.31975632,  6.3849043 ,  1.45052782,
          3.93135892,  2.68895856, 13.38229039,  5.43904725],
        [14.30508614, 10.3849178 ,  9.75462382, 21.95375372,
          1.39888496,  4.19936522,  0.35125783,  1.21400133,
          1.83358404,  3.36844402,  1.88447969, 12.36189374],
        [ 3.61096988, 21.59368428, 24.83449844,  2.86885862,
          9.23000505,  0.15883386,  1.48678172,  1.14051961,
          0.76592526,  7.63893189,  9.09766711, 12.1617363 ],
        [20.91367587, 34.40110457, 14.09912819,  5.48887054,
          8.79958962,  1.57833425,  1.65447826,  9.76441923,
          0.59503013,  7.93007428,  8.12031642,  4.26970939],
        [16.32388169, 27.42979805,  6.68200814, 26.85248617,
          5.04252007,  0.74343145,  1.50504067,  1.28009891,
          3.34891143,  1.66121209,  2.45409744, 15.63786679],
        [19.09561678, 13.57194733, 17.52296538,  2.65548061,
          4.34844982,  1.13818413,  1.68865819,  1.12372975,
          1.25972758,  0.93476238,  4.57816174, 11.67609885],
        [17.37810635, 19.07380631, 12.33440528,  3.76306186,
          0.51396093,  0.9890773 ,  3.55370426,  1.42956698,
          2.70933581,  4.32760509, 15.24477805, 25.68502905]]])
Coordinates:
  * simulation  (simulation) <U26 208B 'WRF_EC-Earth3_r1i1p1f1' 'WRF_MPI-ESM1...
  * month       (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * year        (year) int64 248B 1990 1991 1992 1993 ... 2017 2018 2019 2020
Attributes:
    variable_id:           t2
    extended_description:  Temperature of the air 2m above Earth's surface. T...
    units:                 degC
    data_type:             Gridded
    resolution:            3 km
    frequency:             hourly
    location_subset:       coordinate selection
    approach:              Time
    downscaling_method:    Dynamical
    institution:           UCLA
    grid_mapping:          Lambert_Conformal
    timezone:              America/Los_Angeles

In [24]:
# Pass the weighted F-S sum data for simplicity
ds = weighted_fs_sum

df_list = []
num_values = (
    1  # Selecting the top value for now, persistence statistics calls for top 5
)
for sim in ds.simulation.values:
    for mon in ds.month.values:
        da_i = ds.sel(month=mon, simulation=sim)
        top_xr = da_i.sortby(da_i, ascending=True)[:num_values].expand_dims(
            ["month", "simulation"]
        )
        top_df_i = top_xr.to_dataframe(name="top_values")
        df_list.append(top_df_i)

# Concatenate list together for all months and simulations
top_df = pd.concat(df_list).drop(columns=["top_values"]).reset_index()
top_df

,month,simulation,year
0,1,WRF_EC-Earth3_r1i1p1f1,2006
1,2,WRF_EC-Earth3_r1i1p1f1,2000
2,3,WRF_EC-Earth3_r1i1p1f1,2000
3,4,WRF_EC-Earth3_r1i1p1f1,2011
4,5,WRF_EC-Earth3_r1i1p1f1,2019
5,6,WRF_EC-Earth3_r1i1p1f1,2015
6,7,WRF_EC-Earth3_r1i1p1f1,1994
7,8,WRF_EC-Earth3_r1i1p1f1,1994
8,9,WRF_EC-Earth3_r1i1p1f1,2001
9,10,WRF_EC-Earth3_r1i1p1f1,1993


### Step 4: Generate XMY data outputs 

Generally, the following data is outputted using the TMY months:
- Date & time (UTC)
- Air temperature at 2m [°C]
- Dew point temperature [°C]
- Relative humidity [%]
- Global horizontal irradiance [W/m2]
- Direct normal irradiance [W/m2]
- Diffuse horizontal irradiance [W/m2]
- Downwelling infrared radiation [W/m2]
- Wind speed at 10m [m/s]
- Wind direction at 10m [°]
- Surface air pressure [Pa]

The following function will retrieve all of this data for the designated TMY month and concatenate it together into a pandas dataframe so that we can export it as a csv file. We'll do this next. 

In [30]:
def generate_tmy_data(top_df):
    """Generate typical meteorological year data
    Output will be a list of dataframes per simulation.
    Print statements throughout the function indicate to the user the progress of the computatioconvert_to_local_time   Parameters
    -----------
    top_df: pd.DataFrame
        Table with column values month, simulation, and year
        Each month-sim-yr combo represents the top candidate that has the lowest weighted sum from the FS statistic
    type: str, "tmy" or "xmy"
        Pass in which type of climate profile to generate

    Returns
    --------
    dict of str:pd.DataFrame
        Dictionary in the format of {simulation:TMY corresponding to that simulation}

    """
    ## ================== GET DATA FROM CATALOG ==================
    vars_and_units = {
        "Air Temperature at 2m": "degC",
        "Dew point temperature": "degC",
        "Relative humidity": "[0 to 100]",
        "Instantaneous downwelling shortwave flux at bottom": "W/m2",
        "Shortwave surface downward direct normal irradiance": "W/m2",
        "Shortwave surface downward diffuse irradiance": "W/m2",
        "Instantaneous downwelling longwave flux at bottom": "W/m2",
        "Wind speed at 10m": "m s-1",
        "Wind direction at 10m": "degrees",
        "Surface Pressure": "Pa",
    }

    # Loop through each variable and grab data from catalog
    all_vars_list = []
    print("STEP 1: RETRIEVING HOURLY DATA FROM CATALOG\n")
    for var, units in vars_and_units.items():
        print(f"Retrieving data for {var}", end="... ")
        data_by_var = get_data(
            variable=var,
            resolution="3 km",
            timescale="hourly",
            data_type="Gridded",
            units=units,
            latitude=(stn_lat - 0.05, stn_lat + 0.05),
            longitude=(stn_lon - 0.06, stn_lon + 0.06),
            area_average="No",
            scenario=["Historical Climate", "SSP 3-7.0"],
            time_slice=(start_year, end_year + 1),
        )
        data_by_var = convert_to_local_time(data_by_var)  # convert to local timezone.
        data_by_var = data_by_var.sel(
            {"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")}
        )  # get desired time slice in local time
        data_by_var = get_closest_gridcell(
            data_by_var, stn_lat, stn_lon, print_coords=False
        )  # retrieve only closest gridcell
        data_by_var = data_by_var.sel(
            simulation=data_models
        )  # Subset for only the models that have solar variables

        # Drop unwanted coords
        data_by_var = data_by_var.squeeze().drop(
            ["lakemask", "landmask", "x", "y", "Lambert_Conformal"]
        )

        all_vars_list.append(data_by_var)  # Append to list
        print("complete!")

    # Merge data from all variables into a single xr.Dataset object
    all_vars_ds = xr.merge(all_vars_list)

    ## ================== CONSTRUCT TMY ==================
    print(
        "\nSTEP 2: CALCULATING TYPICAL METEOROLOGICAL YEAR PER MODEL SIMULATION\nProgress bar shows code looping through each month in the year.\n"
    )
    tmy_df_all = {}
    for sim in all_vars_ds.simulation.values:
        df_list = []
        print(f"Calculating TMY for simulation: {sim}")
        for mon in tqdm(np.arange(1, 13, 1)):
            # Get year corresponding to month and simulation combo
            year = top_df.loc[
                (top_df["month"] == mon) & (top_df["simulation"] == sim)
            ].year.item()

            # Select data for unique month, year, and simulation
            data_at_stn_mon_sim_yr = all_vars_ds.sel(
                simulation=sim, time=f"{mon}-{year}"
            ).expand_dims("simulation")

            # Reformat as dataframe
            df_by_mon_sim_yr = data_at_stn_mon_sim_yr.to_dataframe()
            df_by_mon_sim_yr = df_by_mon_sim_yr.reset_index()

            # Reformat time index to remove seconds
            df_by_mon_sim_yr["time"] = pd.to_datetime(
                df_by_mon_sim_yr["time"].values
            ).strftime("%Y-%m-%d %H:%M")
            df_list.append(df_by_mon_sim_yr)

        # Concatenate all DataFrames together
        tmy_df_by_sim = pd.concat(df_list)
        tmy_df_all[sim] = tmy_df_by_sim

    return tmy_df_all  # Return dict of TMY by simulation

In the next cell, we will run the `generate_tmy_data` function which will retrieve, subset, and format the data for each month according to the TMY months for all requested variables. We have included print statements so you can watch the progress for each variable in each month as it builds. 

<span style="color:#FF0000">**Note**: <span style="color:#000000"> This will take time! On the Analytics Engine JupyterHub, this takes approximately 22 minutes. Progress bars will indicate the status of generating the TMY data for each simulation. 

In [ ]:
tmy_data_to_export = generate_tmy_data(top_df)

STEP 1: RETRIEVING HOURLY DATA FROM CATALOG

Retrieving data for Air Temperature at 2m... Data converted to America/Los_Angeles timezone.
complete!
Retrieving data for Dew point temperature... Data converted to America/Los_Angeles timezone.
complete!
Retrieving data for Relative humidity... Data converted to America/Los_Angeles timezone.
complete!
Retrieving data for Instantaneous downwelling shortwave flux at bottom... Data converted to America/Los_Angeles timezone.
complete!
Retrieving data for Shortwave surface downward direct normal irradiance... Data converted to America/Los_Angeles timezone.
complete!
Retrieving data for Shortwave surface downward diffuse irradiance... Data converted to America/Los_Angeles timezone.
complete!
Retrieving data for Instantaneous downwelling longwave flux at bottom... Data converted to America/Los_Angeles timezone.
complete!
Retrieving data for Wind speed at 10m... Data converted to America/Los_Angeles timezone.
complete!
Retrieving data for Wind dir

  0%|          | 0/12 [00:00<?, ?it/s]

Let's observe what the XMY data looks like for one of the simulations:

In [ ]:
simulation = "WRF_EC-Earth3_r1i1p1f1"
tmy_data_to_export[simulation].head(5)

Next, we visualize the XMY data itself.

In [ ]:
tmy_data_to_export[simulation].plot(
    x="time",
    y=[
        "Air Temperature at 2m",
        "Dew point temperature",
        "Relative humidity",
        "Instantaneous downwelling shortwave flux at bottom",
        "Shortwave surface downward direct normal irradiance",
        "Shortwave surface downward diffuse irradiance",
        "Instantaneous downwelling longwave flux at bottom",
        "Wind speed at 10m",
        "Wind direction at 10m",
        "Surface Pressure",
    ],
    title=f"Persisent Extreme Typical Meteorological Year ({simulation})",
    subplots=True,
    figsize=(10, 8),
    legend=True,
)

Lastly, let's export the TMY data below as csv files. There will be a file per simulation downloaded. When utilizing TMY data in your own workflows, we recommend that **all simulations are considered** in your analyses, especially for future scenarios. Note, if you are working with a custom location, please also provide the optional `stn_elev` argument to `write_tmy_file`; if no elevation value is provided, an elevation value of 0.0 is set as the default. 

In [ ]:
for sim, tmy in tmy_data_to_export.items():
    filename = "TMY_{0}_{1}".format(
        stn_name.replace(" ", "_").replace("(", "").replace(")", ""), sim
    ).lower()
    write_tmy_file(
        filename,
        tmy_data_to_export[sim],
        (start_year, end_year),
        stn_name,
        stn_code,
        stn_lat,
        stn_lon,
        stn_state,
        file_ext="epw",
    )